In [9]:
# --- Imports ---
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Scikit-learn imports
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Plotting import
import plotly.graph_objects as go

# Suppress FutureWarnings from scikit-learn for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. Configuration and Setup ---
# Define project paths assuming this notebook is in the 'notebooks/' directory
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Input file for sales data
INPUT_FILENAME = "madrid_sale_properties_processed_1.csv"
INPUT_FILEPATH = PROCESSED_DATA_DIR / INPUT_FILENAME

# --- 2. Load the Data ---
print(f"--- Loading Data from {INPUT_FILEPATH} ---")
df = pd.read_csv(INPUT_FILEPATH)
print("Data loaded successfully. Shape:", df.shape)

# --- 3. Preprocessing and Encoding ---
# This section mirrors the logic in your final training script.

print("\n--- Preprocessing Data ---")

# Define target variable
target_variable = 'price_eur'

# Drop unnecessary/redundant columns. We now drop lat, lon, and distrito.
columns_to_drop = [
    'url', 'property_id', 'scraped_at', 'energy_cert_classification',
    'description', 'superficie_util', 'orientacion_list',
    'distrito', 'latitude', 'longitude' # Dropping these as we will use 'barrio_encoded' as the sole location feature
]
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Drop columns with high NaN percentage (but keep 'amueblado')
NAN_DROP_THRESHOLD = 0.5
COLS_TO_KEEP_REGARDLESS_OF_NAN = ['amueblado']

missing_value_percent = df.isnull().sum() / len(df)
cols_with_high_nan = missing_value_percent[missing_value_percent > NAN_DROP_THRESHOLD].index
cols_to_drop_high_nan = [col for col in cols_with_high_nan if col not in COLS_TO_KEEP_REGARDLESS_OF_NAN]

if cols_to_drop_high_nan:
    df.drop(columns=cols_to_drop_high_nan, inplace=True)
    print(f"Dropped {len(cols_to_drop_high_nan)} columns with >{NAN_DROP_THRESHOLD*100}% missing values: {cols_to_drop_high_nan}")

# Ordinal Encoding
age_map = {
    'Más de 50 años': 0, 'Entre 30 y 50 años': 1, 'Entre 20 y 30 años': 2,
    'Entre 10 y 20 años': 3, 'Entre 5 y 10 años': 4, 'Menos de 5 años': 5
}
condition_map = {'A reformar': 0, 'En buen estado': 1, 'Reformado': 2, 'A estrenar': 3}
df['antiguedad'] = df['antiguedad'].map(age_map)
df['conservacion'] = df['conservacion'].map(condition_map)

# Target Encoding (now only for 'barrio')
df['barrio'] = df['barrio'].fillna('Desconocido')
barrio_map = df.groupby('barrio')[target_variable].mean()
df['barrio'] = df['barrio'].map(barrio_map)
df.rename(columns={'barrio': 'barrio_encoded'}, inplace=True)

# One-Hot Encoding
df = pd.get_dummies(df, columns=['amueblado'], prefix='amueblado', dummy_na=True)

# Isolate features (X) for imputation
features_df = df.drop(columns=[target_variable])

# Iterative Imputation
print("Applying IterativeImputer...")
imputer = IterativeImputer(max_iter=10, random_state=42)
features_imputed_array = imputer.fit_transform(features_df)
features_imputed_df = pd.DataFrame(features_imputed_array, columns=features_df.columns, index=features_df.index)

# --- 4. Feature Scaling ---
print("\n--- Scaling Features ---")
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_imputed_df)
print("Features scaled successfully.")

# --- 5. Elbow Method for Optimal K ---
def find_optimal_k(data, max_k=15):
    """
    Finds the optimal number of clusters (k) for K-Means using the Elbow Method
    and displays an interactive plot.
    """
    print("\n--- Running Elbow Method ---")
    inertias = []
    k_range = range(1, max_k + 1)
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)
        print(f"Inertia for k={k}: {kmeans.inertia_:.2f}")

    # Plotting the elbow curve
    fig = go.Figure(data=go.Scatter(x=list(k_range), y=inertias, mode='lines+markers'))
    fig.update_layout(
        title='<b>Elbow Method for Optimal K</b>',
        xaxis_title='Number of Clusters (K)',
        yaxis_title='Inertia (Within-cluster sum of squares)',
        template='plotly_white'
    )
    print("\nDisplaying elbow plot. Look for the 'elbow' point where the rate of decrease significantly slows.")
    fig.show()

# --- Execute the function ---
# This will now run on the corrected feature set
find_optimal_k(features_scaled, max_k=20)# In your Jupyter Notebook (e.g., model_training.ipynb)

--- Loading Data from d:\Hector\HAB\PFB\streamlit-house-price-prediction\data\processed\madrid_sale_properties_processed_1.csv ---
Data loaded successfully. Shape: (3002, 41)

--- Preprocessing Data ---
Dropped 5 columns with >50.0% missing values: ['energy_consumption_rating', 'energy_emissions_rating', 'energy_consumption_kwh_m2_yr', 'energy_emissions_kg_co2_m2_yr', 'gastos_comunidad_eur']
Applying IterativeImputer...

--- Scaling Features ---
Features scaled successfully.

--- Running Elbow Method ---
Inertia for k=1: 81054.00
Inertia for k=2: 71946.11
Inertia for k=3: 65252.80
Inertia for k=4: 65681.96
Inertia for k=5: 58806.94
Inertia for k=6: 56016.13
Inertia for k=7: 54270.76
Inertia for k=8: 51860.35
Inertia for k=9: 49972.33
Inertia for k=10: 47955.61
Inertia for k=11: 46972.18
Inertia for k=12: 45782.35
Inertia for k=13: 45129.98
Inertia for k=14: 44124.62
Inertia for k=15: 43422.68
Inertia for k=16: 42712.18
Inertia for k=17: 41585.69
Inertia for k=18: 41066.36
Inertia for k